# Step 04a -- Toy-instance sanity check for PPO (diagnostic B.1)

**Why this notebook exists.** The first PPO run (100k timesteps on `escalon_1`)
learned a degenerate policy: 27.6% served vs. 88% for the base policy, very
low occupancy, almost no fleet movement. Before applying any fix (reward
shaping, normalization, hyperparameters -- see `simulacion/README.md`,
diagnosis section), we need to rule out a **structural** bug in the
environment: if PPO cannot learn *anything* on a trivial problem, the issue
is not scale or reward shape, it's something broken in the mechanics
(state/action/reward wiring).

**Deliberately uses the ORIGINAL (unfixed) recipe** -- no `VecNormalize`,
default PPO hyperparameters, `peso_movimiento=0.1`,
`premio_por_persona_entregada=0.0` (both current config defaults, unchanged
from the run that failed) -- on `escalones.escalon_toy` (1 boat, 2 nodes,
small demand, `simulacion/config/instance.yaml`). If this can't learn
anything even here, something structural needs fixing before touching
reward shaping. If it DOES learn something (even a little), the environment
itself is sound, and the full-scale failure is a scale/reward-shape/
hyperparameter problem -- proceed to `04_entrenamiento_rl.ipynb` with the
fixes (VecNormalize, delivery bonus, movement-penalty toggle, entropy/LR/
n_steps), which IS enabled by default in the config from this point on.


In [1]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

import numpy as np
import pandas as pd
import yaml

BASE_DIR = Path.cwd().parent
BB_DIR = (BASE_DIR / "../bergen-boats").resolve()
DEMAND_DIR = (BASE_DIR / "../demand").resolve()
sys.path.insert(0, str(DEMAND_DIR / "src"))
sys.path.insert(0, str(BASE_DIR / "src"))

import masas as demand_masas
import llegadas as demand_llegadas
from entrenamiento import EntornoDemandaAleatoria
from politica_base import asignar_flota

cfg_sim = yaml.safe_load(open(BASE_DIR / "config" / "instance.yaml", encoding="utf-8"))
cfg_bb = yaml.safe_load(open(BB_DIR / "config" / "instance.yaml", encoding="utf-8"))
cfg_demand = demand_masas.cargar_config(DEMAND_DIR / "config" / "instance.yaml")

resumen_masas = pd.read_csv(DEMAND_DIR / "output" / "masas_por_nodo.csv", index_col="id")
intensidad_od = pd.read_csv(DEMAND_DIR / "output" / "matriz_intensidad_od.csv")
poblacion_total_zonas = resumen_masas["poblacion_total"].sum()
conexiones_fuertes = cfg_bb["garantia"]["conexiones_fuertes"]

matriz_tiempos = pd.read_csv(BB_DIR / "02_ruteo_navegable" / "output" / "matriz_tiempos_min.csv", index_col=0)

cfg_escalon = cfg_sim["escalones"]["escalon_toy"]
hora_ini_min, hora_fin_min = cfg_escalon["horas"][0] * 60, cfg_escalon["horas"][1] * 60
nodos_toy = cfg_escalon["nodos"]

# Recompensa ORIGINAL (sin fix) -- explicitamente, no lee usar_vecnormalize/ent_coef/etc.
# del config, porque esos campos representan la propuesta de arreglo (fase B.2+), no lo
# que se quiere probar aca (si el problema es estructural, con la receta que fallo).
cfg_recompensa_original = dict(cfg_sim["recompensa"])
cfg_recompensa_original["premio_por_persona_entregada"] = 0.0

OUT_DIR = BASE_DIR / "output" / "rl_ppo"
OUT_DIR.mkdir(parents=True, exist_ok=True)


def construir_entorno_toy(semilla_entrenamiento):
    return EntornoDemandaAleatoria(
        generar_llegadas_dia_fn=demand_llegadas.generar_llegadas_dia,
        cfg_demand=cfg_demand, intensidad_od=intensidad_od, conexiones_fuertes=conexiones_fuertes,
        poblacion_total_zonas=poblacion_total_zonas, horas=cfg_escalon["horas"],
        porcentaje_poblacion_dia=cfg_escalon["porcentaje_poblacion_dia"],
        semilla_entrenamiento=semilla_entrenamiento,
        matriz_tiempos=matriz_tiempos, nodos=nodos_toy, num_barcos=cfg_escalon["num_barcos"],
        capacidad_barco=cfg_bb["flota"]["capacidad_pasajeros"], nodo_inicial=cfg_bb["flota"]["nodo_inicial"],
        paso_tiempo_min=cfg_sim["paso_tiempo_min"], hora_inicio_min=hora_ini_min, hora_fin_min=hora_fin_min,
        cfg_recompensa=cfg_recompensa_original, unidad_demanda=cfg_sim["unidad_demanda"],
    )


print(f"Toy instance: {cfg_escalon['num_barcos']} boat(s), nodes={nodos_toy}, "
      f"{cfg_escalon['horas']}h, {cfg_escalon['porcentaje_poblacion_dia']*100:.1f}% population/day")


Toy instance: 1 boat(s), nodes=['bryggen', 'kleppesto'], [6, 9]h, 5.0% population/day


## Reference: base policy on the toy instance

In [2]:
def correr_base(semilla):
    env = construir_entorno_toy(semilla_entrenamiento=None)
    obs, info = env.reset(seed=semilla)
    reward_total = 0.0
    while True:
        estado = info["_estado_obj"]
        libres = [b for b in estado.barcos if b.libre]
        decisiones = asignar_flota(libres, estado, matriz_tiempos, env.capacidad_barco, cfg_recompensa_original)
        accion = np.array([
            env.codificar_accion_barco(decisiones[b.id]) if b.libre else 0
            for b in estado.barcos
        ])
        obs, r, terminated, truncated, info = env.step(accion)
        reward_total += r
        if truncated or terminated:
            break
    generadas = env.total_unidades_generadas
    atendidas = len(env.atendidas_historico)
    return reward_total, generadas, atendidas


r_base, gen_base, at_base = correr_base(semilla=42)
print(f"Base policy (seed=42): reward={r_base:.2f}, generated={gen_base}, served={at_base} "
      f"({100*at_base/gen_base:.1f}%)")


Base policy (seed=42): reward=-3694.02, generated=313, served=210 (67.1%)


## Train vanilla PPO (original recipe, small budget)

In [3]:
from stable_baselines3 import PPO
from stable_baselines3.common.env_checker import check_env
from stable_baselines3.common.monitor import Monitor

env_check = construir_entorno_toy(semilla_entrenamiento=cfg_sim["agente"]["entrenamiento"]["semilla_entrenamiento"])
check_env(env_check, warn=True)
print("OK: check_env found no issues on the toy instance either.")

TOTAL_TIMESTEPS_TOY = 20000  # small budget, fast iteration -- this is a sanity check, not a real training run

env_toy = construir_entorno_toy(semilla_entrenamiento=cfg_sim["agente"]["entrenamiento"]["semilla_entrenamiento"])
env_toy_monitor = Monitor(env_toy, filename=str(OUT_DIR / "monitor_toy"))

model_toy = PPO("MlpPolicy", env_toy_monitor, gamma=cfg_sim["agente"]["gamma"], verbose=0)
model_toy.learn(total_timesteps=TOTAL_TIMESTEPS_TOY)
print("Training done.")


OK: check_env found no issues on the toy instance either.


Training done.


## Verdict: did it learn anything?

In [4]:
# Se lee el CSV del monitor DIRECTAMENTE (no con `load_results(OUT_DIR)`, que junta
# TODOS los *.monitor.csv de la carpeta -- mezclaria esta corrida de juguete con la
# corrida completa de 04_entrenamiento_rl.ipynb si ya se corrio antes). El formato del
# Monitor de SB3 es una linea de comentario JSON + encabezado real "r,l,t".
monitor_toy_path = OUT_DIR / "monitor_toy.monitor.csv"
df_monitor_toy = pd.read_csv(monitor_toy_path, skiprows=1)
y = df_monitor_toy["r"].values

n = len(y)
primeros = y[: max(1, n // 5)]
ultimos = y[-max(1, n // 5):]
print(f"{n} episodes trained. Mean reward -- first {len(primeros)}: {primeros.mean():.2f}, "
      f"last {len(ultimos)}: {ultimos.mean():.2f}")

mejora = ultimos.mean() - primeros.mean()
print(f"Improvement (last - first): {mejora:+.2f}")

if mejora > 0.05 * abs(primeros.mean()):
    print("\nVERDICT: PPO learns SOMETHING on the toy instance -- the environment is "
          "structurally sound. Proceeding to 04_entrenamiento_rl.ipynb with the fixes "
          "(VecNormalize, delivery bonus, movement-penalty toggle, entropy/LR/n_steps).")
else:
    print("\nVERDICT: PPO does NOT show meaningful improvement even on the toy instance. "
          "STOP -- this points to something structural (reward/observation/action wiring), "
          "not just scale or reward shape. Report before applying any further fix.")


227 episodes trained. Mean reward -- first 45: -8845.11, last 45: -8705.65
Improvement (last - first): +139.46

VERDICT: PPO does NOT show meaningful improvement even on the toy instance. STOP -- this points to something structural (reward/observation/action wiring), not just scale or reward shape. Report before applying any further fix.


In [5]:
def correr_agente(semilla):
    env = construir_entorno_toy(semilla_entrenamiento=None)
    obs, info = env.reset(seed=semilla)
    reward_total = 0.0
    while True:
        accion, _ = model_toy.predict(obs, deterministic=True)
        obs, r, terminated, truncated, info = env.step(accion)
        reward_total += r
        if truncated or terminated:
            break
    generadas = env.total_unidades_generadas
    atendidas = len(env.atendidas_historico)
    return reward_total, generadas, atendidas


r_agente, gen_agente, at_agente = correr_agente(semilla=42)
print(f"Trained agent (seed=42, same demand as the base-policy reference above):")
print(f"  reward={r_agente:.2f}, generated={gen_agente}, served={at_agente} ({100*at_agente/gen_agente:.1f}%)")
print(f"\nFor comparison, base policy: reward={r_base:.2f}, served={at_base} ({100*at_base/gen_base:.1f}%)")


Trained agent (seed=42, same demand as the base-policy reference above):
  reward=-13218.86, generated=313, served=0 (0.0%)

For comparison, base policy: reward=-3694.02, served=210 (67.1%)
